In [1]:
import re
import glob,os
import numpy as np
import pandas as pd

In [2]:
#download bertscore
import evaluate
bertscore = evaluate.load("./bertscore")

In [3]:
#check if GPT's output meet the format's requirement by computing the length of splitted output by '|' which is supposed to be 9
def check_output(all_answers):
    for i in range(len(all_answers)):
        standard = all_answers['Manual_answer'][i]
        GPT = all_answers['GPT_answer'][i]
        filename = all_answers['filename'][i]
        if GPT == GPT:
            standard_split = standard.split('\n')
            GPT_split = GPT.split('\n')
            for each_paragraph in standard_split:
                each_paragraph = each_paragraph.split('|')
                each_paragraph = list(filter(None, each_paragraph))
                length = len(each_paragraph)
                if length!=9:
                    print(filename,i,each_paragraph,length)
            for each_paragraph in GPT_split:
                each_paragraph_s = each_paragraph.split('|')
                each_paragraph_s = list(filter(None, each_paragraph_s))
                length = len(each_paragraph_s)
                if length!=9:
                    #print(filename,i,each_paragraph)
                    print(filename,length,each_paragraph_s[0])

In [4]:
# compute overall similarities
def compute_overall_sim(all_answers):
    for i in range(len(all_answers)):
        index = all_answers['Index'][i]
        standard = all_answers['Manual_answer'][i]
        GPT = all_answers['GPT_answer'][i]
        if standard == '2022-zhou-et-al-inherently-chiral-cages-via-hierarchical-desymmetrization.txt':
            with open(standard,'r') as f:
                read_from_txt = f.read()
            standard = read_from_txt
            print('read from file')
        else:
            pass
        if standard == standard:
            predictions = [GPT]
            references = [standard]
            results = bertscore.compute(predictions=predictions, references=references, lang="en", model_type = "distilbert-base-uncased")
            recall = results['recall']
            all_answers.loc[i, "overall_recall"] = recall[0]
            print(recall[0],index)
    return all_answers

In [5]:
def get_monomers(text):
    text_split = text.split('\n')
    matrix = []
    for each_paragraph in text_split:
        each_paragraph = each_paragraph.split('|')
        each_paragraph = list(filter(None, each_paragraph))
        length = len(each_paragraph)
        if length == 9:
            monomer1 = each_paragraph[-4]
            monomer2 = each_paragraph[-2]
            synthesis1 = each_paragraph[-3]
            synthesis2 = each_paragraph[-1]
            
            matrix.append([monomer1+';'+monomer2,synthesis1+';'+synthesis2])
        else:
            print('error',length,each_paragraph)
    matrix = list(map(list, zip(*matrix)))
    return matrix

In [6]:
def extract_number_plus_number(text):
    # Regular expression pattern to match text in the form of [number+number]
    pattern = r'\[\d+\+\d+\]'
    new_text = ''.join(text.split())
    matches = re.findall(pattern, new_text)
    if matches == []:
        matches = [text]
    return matches

In [7]:
def get_others(text):
    text_split = text.split('\n')
    matrix = []
    for each_paragraph in text_split:
        each_paragraph = each_paragraph.split('|')
        each_paragraph = list(filter(None, each_paragraph))
        length = len(each_paragraph)
        if length == 9:
            each_paragraph_ = [each_paragraph[i] for i in [0,2,3,4]]
            topology = extract_number_plus_number(each_paragraph[1])
            topology = ' '.join(topology)
            if topology == '':
                topology = 'None'
            each_paragraph_.append(topology)
            matrix.append(each_paragraph_)
        else:
            print('error',length,each_paragraph)
    matrix = list(map(list, zip(*matrix)))
    return matrix

In [8]:
# compute similarities of monomers and their synthesis route
def compute_monomer_sim(all_answers):
    for i in range(len(all_answers)):
        standard = all_answers['Manual_answer'][i]
        GPT = all_answers['GPT_answer'][i]
        # the content in "2022-zhou-et-al ..." cannot be saved in excel files due to the limitation of characters, so read from txt files specially
        if standard == '2022-zhou-et-al-inherently-chiral-cages-via-hierarchical-desymmetrization.txt':
            with open(standard,'r') as f:
                read_from_txt = f.read()
            standard = read_from_txt
        else:
            pass
        standard_matrix = get_monomers(standard)
        GPT_matrix = get_monomers(GPT)
        standard_matrix = ['|'.join(each_part) for each_part in standard_matrix]
        GPT_matrix = ['|'.join(each_part) for each_part in GPT_matrix]
        results = bertscore.compute(predictions=GPT_matrix, references=standard_matrix, lang="en", model_type = "distilbert-base-uncased")
        recall = str(results['recall'])
        all_answers.loc[i, "recall_monomer"] = recall
        name1,name2 = standard_matrix[0],GPT_matrix[0]
        print(i,recall,name1,name2)
    return all_answers

In [9]:
#compute similarities of other sections
def compute_others_sim(all_answers):
    for i in range(len(all_answers)):
        standard = all_answers['Manual_answer'][i]
        GPT = all_answers['GPT_answer'][i]
        # the content in "2022-zhou-et-al ..." cannot be saved in excel files due to the limitation of characters, so read from txt files specially
        if standard == '2022-zhou-et-al-inherently-chiral-cages-via-hierarchical-desymmetrization.txt':
            with open(standard,'r') as f:
                read_from_txt = f.read()
            standard = read_from_txt
        else:
            pass
        standard_matrix = get_others(standard)
        GPT_matrix = get_others(GPT)
        standard_matrix = ['|'.join(each_part) for each_part in standard_matrix]
        GPT_matrix = ['|'.join(each_part) for each_part in GPT_matrix]
        results = bertscore.compute(predictions=GPT_matrix, references=standard_matrix, lang="en", model_type = "distilbert-base-uncased")
        others_recall = str(results['recall'])
        all_answers.loc[i, "recall_others"] = others_recall
        print(standard_matrix[1],GPT_matrix[1],others_recall)
    return all_answers

In [10]:
all_answers = pd.read_excel("Tabular Informations without bertscore.xlsx")[:5]
check_output(all_answers)
all_answers = compute_overall_sim(all_answers)
all_answers = compute_monomer_sim(all_answers)
all_answers = compute_others_sim(all_answers)
print(all_answers)
all_answers.to_excel('test.xlsx')

0.892945408821106 1.0
0.9249517321586609 2.0
0.9107215404510498 3.0
0.9158487319946289 4.0
0.9201462864875793 5.0
0 [0.8377740979194641, 0.8220938444137573] 1;2  Triamine 1 ; Salicylic dialdehyde 2 
1 [0.9766533970832825, 0.6662696599960327] phloroglu-cinol;4,6-Dichloro-3-methylisoxazolo[4,5- c]-pyridine Phloroglucinol;4,6-Dichloro-3-methylisoxazolo[4,5-c]pyridine
2 [0.9459102153778076, 0.9419347047805786] phloroglucinol;intermediate 3  phloroglucinol 1 ; cyanuric chloride 2 
3 [0.9999999403953552, 0.9233015775680542] 1,3,5 -tri-(4-formylphenyl) benzene;1,5 -pentanediamine  1,3,5-tri-(4-formylphenyl) benzene ; 1,5-pentanediamine 
4 [0.9999999403953552, 0.8148008584976196] 2,7,14-Triaminotriptycene;4,6-Dihydroxy-5-methyl-1,3-diformyl benzene  2,7,14-Triaminotriptycene ; 4,6-Dihydroxy-5-methyl-1,3-diformyl benzene 
None  None  [0.881901204586029, 1.0000001192092896, 1.0000001192092896, 0.9394999146461487, 0.6537303924560547]
668972 CCDC-668972 [0.9999998807907104, 0.9628136157989502, 1.0

In [43]:
all_answers

,Index,filename,title,authors,affiliations,Manual_answer,GPT_answer,appendix,Similarity of Total Text,Similarity of Name,Similarity of Synthesis,Similarity of BET,Similarity of CCDC,Similarity of Topology,Similarity of Monomer's Name,Similarity of Monomer's Synthesis,overall_recall,recall_monomer,recall_others
0,1.0,level1\2008---Chem Comm.csv,One-pot synthesis of a shape-persistent endo-f...,Michael Mastalerz,"Ulm University, Institute of Organic Chemistry...",cage compound 3|[4+6]|None|None|A solution of ...,"| Adamantoid nanocage compound 3 | ""The rea...",NaN,0.892945,0.881901,0.939500,1.000000,1.000000,0.653730,0.837774,0.822094,0.892945,"[0.8377740979194641, 0.8220938444137573]","[0.881901204586029, 1.0000001192092896, 1.0000..."
1,2.0,level1\2008-Eur J Org Chem -Ferrini.csv,Synthesis of Isoxazolopyridobicyclooxacalix[4]...,"Serena Ferrini,[a] Stefania Fusi,[a] Gianluca ...","Dipartimento di Chimica, Universit degli Stu...",Isoxazolopyridobicyclooxacalix[4]arenes|[2+3]|...,|Isoxazolopyridobicyclooxacalix[4]arenes|A new...,NaN,0.924952,1.000000,0.467224,1.000000,0.962814,0.709062,0.976653,0.666270,0.924952,"[0.9766533970832825, 0.6662696599960327]","[0.9999998807907104, 0.9628136157989502, 1.000..."
2,3.0,level1\2010-Chemistry A European J - Wang - Ve...,Versatile Anion Cp Interactions between Halide...,"De-Xian Wang,*[a]Qi-Qiang Wang,[a]Yuchun Han,[...",Beijing National Laboratory for Molecular Scie...,"cage molecule 4|[2+3]|734866, 734867, 734868|N...",| bis(tetraoxacalix[2]arene[2]triazine) (4) | ...,NaN,0.910722,0.754178,0.939581,1.000000,0.821137,0.606414,0.945910,0.941935,0.910722,"[0.9459102153778076, 0.9419347047805786]","[0.754177987575531, 0.8211369514465332, 1.0000..."
3,4.0,level1\2011----Chem Comm---Selective Gas Sorpt...,Selective Gas Sorption in a [2+3] Propeller ...,"Shan Jiang, John Bacsa, Xiaofeng Wu, James T. ...",Department of Chemistry and Centre for Materia...,CC6|[2+3]|821249|N2 sorption measurements at 7...,"| CC6 | ""This molecule is synthesized by a one...",NaN,0.915849,1.000000,0.995390,0.941887,1.000000,1.000000,1.000000,0.923302,0.915849,"[0.9999999403953552, 0.9233015775680542]","[1.0, 1.0, 0.9418870210647583, 0.9953902959823..."
4,5.0,level1\2012 ---Chem. Commun---A shape-persiste...,A shape-persistent exo-functionalized [4+6] im...,"Markus W. Schneider,aHans-Jochen Siegfried Hau...",aInstitute of Organic Chemistry II & Advanced ...,cage compound 5|[4+6]|None|Theamorphous materi...,"| Cage Compound 5 | ""Here, we will present a n...",NaN,0.920146,1.000000,0.971945,0.885461,1.000000,1.000000,1.000000,0.814801,0.920146,"[0.9999999403953552, 0.8148008584976196]","[1.000000238418579, 1.0000001192092896, 0.8854..."
